### Imports des paquets utiles

In [1]:
!pip3 uninstall --yes torch torchaudio torchvision torchtext torchdata
!pip3 install torch torchaudio torchvision torchtext torchdata

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, aggr

Found existing installation: torch 2.12.0
Uninstalling torch-2.12.0:
  Successfully uninstalled torch-2.12.0
Found existing installation: torchaudio 2.11.0
Uninstalling torchaudio-2.11.0:
  Successfully uninstalled torchaudio-2.11.0
Found existing installation: torchvision 0.27.0
Uninstalling torchvision-0.27.0:
  Successfully uninstalled torchvision-0.27.0
Found existing installation: torchtext 0.18.0
Uninstalling torchtext-0.18.0:
  Successfully uninstalled torchtext-0.18.0
Found existing installation: torchdata 0.11.0
Uninstalling torchdata-0.11.0:
  Successfully uninstalled torchdata-0.11.0
  Using cached torch-2.12.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (31 kB)
  Using cached torchaudio-2.11.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (6.9 kB)
  Using cached torchvision-0.27.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (5.5 kB)
  Using cached torchtext-0.18.0-cp312-cp312-manylinux1_x86_64.whl.metadata (7.9 kB)
  Using cached torchdata-0.11.0-py3-none-any.whl.met

### étape 1 : Node Embeddings (Base GNN)

In [2]:
class BaseGNN(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        return x # Z matrix

### étape 2: Differentiable Motif Proposal

In [10]:
class MotifProposal(nn.Module):
    def __init__(self, latent_dim: int, tau: float = 1.0):
        super().__init__()
        self.tau = tau

        # On remplace nn.Bilinear par un simple paramètre de poids W de taille [d, d]
        self.W = nn.Parameter(torch.Tensor(latent_dim, latent_dim))

        # Initialisation des poids (Xavier/Glorot est standard)
        nn.init.xavier_uniform_(self.W)

    def forward(self, Z: torch.Tensor, num_motifs_per_node: int = 1):
        """
        Z: [N, d] node embeddings
        Returns:
            probs: [N, N] - matrice d'assignation douce
        """
        # Multiplication matricielle optimisée : (Z @ W) @ Z.T
        # Z est [N, d], W est [d, d] -> (Z @ W) donne [N, d]
        # (N, d) @ (d, N) -> donne un tenseur [N, N] directement
        logits = (Z @ self.W) @ Z.t()

        # Gumbel-Softmax pour obtenir un échantillonnage différentiable
        probs = F.gumbel_softmax(logits, tau=self.tau, hard=False, dim=-1)

        return probs

### Motifs Encoding

In [11]:
class DeepSetsEncoder(nn.Module):
    """
    Encodeur invariant par permutation (DeepSets) pour les motifs découverts.
    """
    def __init__(self, input_dim: int, hidden_dim: int, output_dim: int,
                 aggregator_type: str = 'sum', dropout: float = 0.0):
        super().__init__()

        self.psi = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh()
        )

        if aggregator_type == 'max':
            self.aggregator = aggr.MaxAggregation()
        elif aggregator_type == 'mean':
            self.aggregator = aggr.MeanAggregation()
        elif aggregator_type == 'sum':
            self.aggregator = aggr.SumAggregation()
        else:
            raise ValueError(f"Unknown aggregator_type: {aggregator_type}")

        self.phi = nn.Sequential(
            nn.Linear(hidden_dim, output_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(output_dim, output_dim)
        )

    def forward(self, x: torch.Tensor, batch_index: torch.Tensor) -> torch.Tensor:
        h = self.psi(x)
        h_agg = self.aggregator(h, batch_index)
        y = self.phi(h_agg)
        return y

### STEP 4: Differentiable Motif Discovery (DMD) Pipeline

In [13]:
class DMDModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim, motif_out_dim, num_classes):
        super().__init__()
        self.gnn = BaseGNN(input_dim, hidden_dim, latent_dim)
        self.motif_proposal = MotifProposal(latent_dim, tau=0.5)
        self.motif_encoder = DeepSetsEncoder(latent_dim, hidden_dim, motif_out_dim)

        # Classifieur final combinant l'embedding du nœud et l'embedding de son motif
        self.classifier = nn.Linear(latent_dim + motif_out_dim, num_classes)

    def forward(self, x, edge_index):
        N = x.size(0)

        # 1. Base GNN
        Z = self.gnn(x, edge_index)

        # 2. Motif Proposal (Soft assignment matrix [N, N])
        # Chaque ligne i est la probabilité que les autres nœuds appartiennent au motif de i
        P = self.motif_proposal(Z)

        # 3. Aggregation (Approximation différentiable du DeepSets)
        # Plutôt que de créer un index de batch strict (qui brise la différentiabilité si on filtre),
        # on pondère les embeddings par les probabilités P.
        # h_motif_i = Encodeur( \sum_j P_ij * Psi(Z_j) )

        # Application de Psi sur tous les noeuds
        psi_Z = self.motif_encoder.psi(Z) # [N, hidden_dim]

        # Agrégation pondérée par P (Soft SumAggregation)
        # P est de taille [N, N], psi_Z de taille [N, hidden_dim]
        # h_agg[i] = sum_j P[i,j] * psi_Z[j]
        h_agg = torch.matmul(P, psi_Z) # [N, hidden_dim]

        # Application de Phi
        motif_embeddings = self.motif_encoder.phi(h_agg) # [N, motif_out_dim]

        # 4. Concaténation et Classification
        out_features = torch.cat([Z, motif_embeddings], dim=-1)
        logits = self.classifier(out_features)

        return logits, P # On retourne P pour potentiellement appliquer une Sparsity Loss

In [14]:
import torch
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
import torch_geometric.transforms as T

# Chargement du dataset Cora (Classification de nœuds)
dataset = Planetoid(root='/tmp/Cora', name='Cora', transform=T.NormalizeFeatures())
data = dataset[0]

# Vérification des dimensions
input_dim = dataset.num_node_features
num_classes = dataset.num_classes
print(f"Nodes: {data.num_nodes}, Edges: {data.edge_index.size(1)}")
print(f"Features: {input_dim}, Classes: {num_classes}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data = data.to(device)

Nodes: 2708, Edges: 10556
Features: 1433, Classes: 7


In [15]:
class DMDLoss(nn.Module):
    def __init__(self, sparsity_weight: float = 0.01):
        super().__init__()
        self.task_loss = nn.CrossEntropyLoss()
        self.sparsity_weight = sparsity_weight

    def forward(self, logits, target, P_matrix, mask):
        """
        logits: Prédictions [N, C]
        target: Vraies classes [N]
        P_matrix: Matrice d'assignation des motifs [N, N]
        mask: Masque d'entraînement (train_mask, val_mask, test_mask)
        """
        # 1. Loss de classification standard (sur les noeuds masqués)
        loss_task = self.task_loss(logits[mask], target[mask])

        # 2. Sparsity Loss (L1 regularization sur P pour forcer des motifs locaux et de petite taille)
        # On veut que chaque nœud ne sélectionne que peu de nœuds pour former son motif
        loss_sparsity = torch.norm(P_matrix, p=1) / (P_matrix.size(0) * P_matrix.size(1))

        # 3. (Future) Orthogonality Loss : pour éviter de découvrir le même motif partout
        # loss_ortho = torch.norm(torch.matmul(P_matrix, P_matrix.T) - torch.eye(N))

        total_loss = loss_task + self.sparsity_weight * loss_sparsity
        return total_loss, loss_task, loss_sparsity

In [16]:
def train_dmd(model, data, optimizer, criterion):
    model.train()
    optimizer.zero_grad()

    # Forward pass: on récupère les prédictions ET la distribution des motifs
    logits, P_matrix = model(data.x, data.edge_index)

    # Calcul de la loss
    loss, l_task, l_sparse = criterion(logits, data.y, P_matrix, data.train_mask)

    # Backward pass
    loss.backward()
    optimizer.step()

    return loss.item(), l_task.item(), l_sparse.item()

@torch.no_grad()
def evaluate_dmd(model, data, criterion, mask):
    model.eval()
    logits, P_matrix = model(data.x, data.edge_index)

    # Calcul de la loss
    loss, l_task, l_sparse = criterion(logits, data.y, P_matrix, mask)

    # Calcul de l'Accuracy
    pred = logits[mask].argmax(dim=-1)
    correct = pred.eq(data.y[mask]).sum().item()
    accuracy = correct / mask.sum().item()

    return loss.item(), accuracy

In [17]:
# Hyperparamètres
hidden_dim = 64
latent_dim = 32
motif_out_dim = 32
epochs = 200
lr = 0.005
weight_decay = 5e-4

# Instanciation (DMDModel a été défini dans l'étape précédente)
model = DMDModel(input_dim, hidden_dim, latent_dim, motif_out_dim, num_classes).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
criterion = DMDLoss(sparsity_weight=0.05)

print("Début de l'entraînement de DMD...")
best_val_acc = 0
test_acc_at_best_val = 0

for epoch in range(1, epochs + 1):
    # Entraînement
    train_loss, t_task, t_sparse = train_dmd(model, data, optimizer, criterion)

    # Évaluation
    val_loss, val_acc = evaluate_dmd(model, data, criterion, data.val_mask)
    test_loss, test_acc = evaluate_dmd(model, data, criterion, data.test_mask)

    # Tracking du meilleur modèle
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        test_acc_at_best_val = test_acc

    if epoch % 20 == 0 or epoch == 1:
        print(f"Epoch {epoch:03d} | Train Loss: {train_loss:.4f} (Task: {t_task:.4f}, Sparse: {t_sparse:.4f}) | "
              f"Val Acc: {val_acc:.4f} | Test Acc: {test_acc:.4f}")

print(f"\n--- Fin de l'entraînement ---")
print(f"Meilleure Val Acc: {best_val_acc:.4f}")
print(f"Test Acc (au meilleur Val): {test_acc_at_best_val:.4f}")

Début de l'entraînement de DMD...
Epoch 001 | Train Loss: 1.9465 (Task: 1.9465, Sparse: 0.0004) | Val Acc: 0.3200 | Test Acc: 0.3630
Epoch 020 | Train Loss: 1.6662 (Task: 1.6661, Sparse: 0.0004) | Val Acc: 0.6200 | Test Acc: 0.6320
Epoch 040 | Train Loss: 0.6436 (Task: 0.6436, Sparse: 0.0004) | Val Acc: 0.7280 | Test Acc: 0.7460
Epoch 060 | Train Loss: 0.1573 (Task: 0.1573, Sparse: 0.0004) | Val Acc: 0.7800 | Test Acc: 0.7920
Epoch 080 | Train Loss: 0.0528 (Task: 0.0528, Sparse: 0.0004) | Val Acc: 0.7560 | Test Acc: 0.7650
Epoch 100 | Train Loss: 0.0840 (Task: 0.0840, Sparse: 0.0004) | Val Acc: 0.7220 | Test Acc: 0.7210
Epoch 120 | Train Loss: 0.0375 (Task: 0.0375, Sparse: 0.0004) | Val Acc: 0.7560 | Test Acc: 0.7820
Epoch 140 | Train Loss: 0.0295 (Task: 0.0295, Sparse: 0.0004) | Val Acc: 0.7500 | Test Acc: 0.7850
Epoch 160 | Train Loss: 0.0248 (Task: 0.0248, Sparse: 0.0004) | Val Acc: 0.7360 | Test Acc: 0.7680
Epoch 180 | Train Loss: 0.0242 (Task: 0.0242, Sparse: 0.0004) | Val Acc: 0.